In [ ]:
import os
os.environ["OPENCV_LOG_LEVEL"] = "FATAL"
import cv2
import threading
from socket import *
import time
from flask import Flask, Response
import logging
from LOBOROBOT import LOBOROBOT

In [ ]:
# ================= 1. 初始化硬件 =================
print("初始化底盘与摄像头...")

bot = LOBOROBOT()
bot.t_stop(0)

# 初始化云台角度
PAN_CH, TILT_CH = 10, 9
current_pan, current_tilt = 80, 0

bot.set_servo_angle(PAN_CH, current_pan)
bot.set_servo_angle(TILT_CH, current_tilt)

In [ ]:
# ================= 2. 视频流全局机制 =================
VIDEO_WIDTH = 1280
VIDEO_HEIGHT = 720
VIDEO_FPS = 20

STREAM_JPEG_QUALITY = 82     # 实时视频流压缩质量
CAPTURE_JPEG_QUALITY = 95    # 拍照图片压缩质量

global_frame = None          # 给视频流使用的 JPEG 数据
latest_raw_frame = None      # 给拍照使用的原始图像帧
frame_lock = threading.Lock()


def create_camera():
    """
    创建摄像头对象
    """
    gstreamer_pipeline = (
        f"libcamerasrc ! video/x-raw, width={VIDEO_WIDTH}, height={VIDEO_HEIGHT}, framerate={VIDEO_FPS}/1 ! "
        "videoconvert ! video/x-raw, format=BGR ! appsink drop=true max-buffers=1"
    )

    cap = cv2.VideoCapture(gstreamer_pipeline, cv2.CAP_GSTREAMER)

    if cap.isOpened():
        print(f"✅ 视频摄像头打开成功：{VIDEO_WIDTH}x{VIDEO_HEIGHT}@{VIDEO_FPS}fps")
    else:
        print("❌ 视频摄像头打开失败")

    return cap


def video_capture_thread():
    """
    摄像头采集线程：
    1. 持续读取摄像头画面
    2. 保存一份原始帧 latest_raw_frame 给拍照接口使用
    3. 编码一份 JPEG global_frame 给视频流接口使用
    """
    global global_frame, latest_raw_frame

    cap = create_camera()

    while True:
        if cap is None or not cap.isOpened():
            print("▶️ 正在重新打开视频摄像头...")
            cap = create_camera()
            time.sleep(1.0)
            continue

        ret, frame = cap.read()

        if not ret:
            time.sleep(0.02)
            continue

        # 摄像头倒装时使用，等效旋转 180°
        frame = cv2.flip(frame, -1)

        # 保存原始帧，用于拍照时高质量编码
        with frame_lock:
            latest_raw_frame = frame.copy()

        # 编码成 JPEG，用于实时视频流
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), STREAM_JPEG_QUALITY]
        ret_enc, buffer = cv2.imencode(".jpg", frame, encode_param)

        if ret_enc:
            with frame_lock:
                global_frame = buffer.tobytes()

        time.sleep(0.001)


t_cam = threading.Thread(target=video_capture_thread)
t_cam.daemon = True
t_cam.start()

In [ ]:
# ================= 3. Flask 服务器 =================
app = Flask(__name__)

log = logging.getLogger("werkzeug")
log.setLevel(logging.ERROR)


# 接口1：实时视频流
@app.route("/mycamera")
def video_feed():
    def generate_stream():
        while True:
            with frame_lock:
                jpeg = global_frame

            if jpeg is not None:
                yield (
                    b"--frame\r\n"
                    b"Content-Type: image/jpeg\r\n\r\n" + jpeg + b"\r\n"
                )

            time.sleep(0.04)

    return Response(
        generate_stream(),
        mimetype="multipart/x-mixed-replace; boundary=frame"
    )


# 接口2：拍照接口
@app.route("/capture")
def capture_photo():
    global latest_raw_frame

    print("📸 收到远程拍照请求，截取当前视频帧...")

    try:
        # 拍照前停车，减少画面抖动
        bot.t_stop(0)

        with frame_lock:
            if latest_raw_frame is None:
                print("❌ 拍照失败：当前没有可用视频帧")
                return "拍照失败：当前没有可用视频帧", 500

            frame = latest_raw_frame.copy()

        # 使用更高 JPEG 质量重新编码当前帧
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), CAPTURE_JPEG_QUALITY]
        ret_enc, buffer = cv2.imencode(".jpg", frame, encode_param)

        if not ret_enc:
            print("❌ 拍照失败：图像编码失败")
            return "拍照失败：图像编码失败", 500

        jpeg_bytes = buffer.tobytes()

        print("✅ 当前视频帧截取成功，开始发送给上位机")

        return Response(
            jpeg_bytes,
            mimetype="image/jpeg",
            headers={
                "Content-Disposition": "inline; filename=capture.jpg"
            }
        )

    except Exception as e:
        print("❌ 拍照接口异常：", str(e))
        return f"Error: {str(e)}", 500

In [ ]:
# ================= 4. 网络启动 =================
def get_ip_address():
    try:
        s = socket(AF_INET, SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except:
        return "127.0.0.1"


ip = get_ip_address()

print("✅ 树莓派就绪！")
print(f"✅ 视频流地址: http://{ip}:8080/mycamera")
print(f"✅ 拍照下载地址: http://{ip}:8080/capture")


def run_flask():
    app.run(
        host="0.0.0.0",
        port=8080,
        threaded=True,
        use_reloader=False
    )


t_flask = threading.Thread(target=run_flask)
t_flask.daemon = True
t_flask.start()

In [ ]:
# ================= 5. UDP 控制循环 =================
udp_server = socket(AF_INET, SOCK_DGRAM)
udp_server.bind(("0.0.0.0", 2001))

speed = 50

try:
    while True:
        data_recv, addr = udp_server.recvfrom(1024)
        cmd = data_recv.decode("utf-8").strip()

        if cmd == "UP":
            bot.t_up(speed, 0)

        elif cmd == "DOWN":
            bot.t_down(speed, 0)

        elif cmd == "LEFT_MOVE":
            bot.moveLeft(speed, 0)

        elif cmd == "RIGHT_MOVE":
            bot.moveRight(speed, 0)

        elif cmd == "TURN_L":
            bot.turnLeft(speed, 0)

        elif cmd == "TURN_R":
            bot.turnRight(speed, 0)

        elif cmd == "UP_LEFT":
            bot.forward_Left(speed, 0)

        elif cmd == "UP_RIGHT":
            bot.forward_Right(speed,0)

        elif cmd == "DOWN_LEFT":
            bot.backward_Left(speed,0)

        elif cmd == "DOWN_RIGHT":
            bot.backward_Right(speed,0)

        elif cmd == "STOP":
            bot.t_stop(0)

        elif cmd == "CAM_UP":
            current_tilt = max(0, current_tilt - 10)
            bot.set_servo_angle(TILT_CH, current_tilt)

        elif cmd == "CAM_DOWN":
            current_tilt = min(180, current_tilt + 10)
            bot.set_servo_angle(TILT_CH, current_tilt)

        elif cmd == "CAM_LEFT":
            current_pan = min(180, current_pan + 10)
            bot.set_servo_angle(PAN_CH, current_pan)

        elif cmd == "CAM_RIGHT":
            current_pan = max(0, current_pan - 10)
            bot.set_servo_angle(PAN_CH, current_pan)

except KeyboardInterrupt:
    print("终止")

finally:
    bot.t_stop(0)
    udp_server.close()